# Logical qubit counts

The width model follows a typed decomposition and visits its children sequentially. At each call, it adds the callee's peak to live registers that bypass the call. Independent connection components can reuse space. When decomposition is unavailable, the estimate takes the maximum of the signature width and declared callee widths. This model accounts for each child at its full peak for the duration of the call. The physical binder separately checks work ownership and the assigned wires.

In [ ]:
from collections import Counter
from dataclasses import replace
from io import BytesIO

from IPython.display import Image, Markdown, SVG, display
from matplotlib import pyplot
from sympy import Symbol

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.qubit_count_protocol_catalog import (
    interior_width_call_graph, interior_width_connections, interior_width_model,
    interior_workspace_fixture, interior_workspace_signature,
)
from flagquantum.circuit import Circuit
from flagquantum.compiler._component_trace.program_logical_width import query_program_logical_width
from flagquantum.compiler._component_trace.width_schedule import estimate_connection_width
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.component_score import component_register_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score

## Interior allocation

The signature contains two through registers of width `n`. Allocating a third word between them makes the peak `3*n`. The diagram shows the symbolic interface, followed by the stored call schedule at width two.

In [ ]:
n = Symbol("n", positive=True, integer=True)
signature = interior_workspace_signature(n)
assert signature.n_qubits() == 2 * n
display(SVG(data=field_signature_svg([("Interior allocation", signature, {})])))
ports = ", ".join(f"`{reg.name}` is a through word of width `{reg.bitsize}`" for reg in signature)
display(Markdown(ports + "."))

program = interior_workspace_fixture(2)

def style_step(function, step):
    if function.name == "swap_word":
        label = "Swap"
    else:
        opcode = function.blocks[0].operations[0].opcode
        label = {"allocate_register": "Allocate", "free_register": "Free"}[opcode]
    return replace(step, name=label)

score = component_register_score(program, style=style_step)
figure, _ = draw_register_score(register_score_data(score), unit_to_inches=0.95)
with BytesIO() as output:
    figure.savefig(output, format="png", dpi=120, bbox_inches="tight", pad_inches=0.15)
    display(Image(data=output.getvalue()))
pyplot.close(figure)

In [ ]:
widths = {}
peak = interior_width_model(n, cache=widths)
assert peak == 3 * n
peak

In [ ]:
graph = interior_width_call_graph(n)

def width_label(name):
    return {
        "label": f"{name}\nLogical qubits {widths[name]}",
        "fontname": "Helvetica", "shape": "box", "style": "rounded,filled",
        "fillcolor": "#eef5f8", "margin": "0.18,0.10",
    }

dot = export_call_graph_dot(graph.root, graph.direct_calls, node_properties=width_label)
display(SVG(data=render_call_graph_svg(dot)))

## Query the decomposition

The symbolic connection graph and a finite typed Program expose the same peak. Inspect the bound primitive stream separately from the model.

In [ ]:
steps, edges = interior_width_connections(n, widths.__getitem__)
assert estimate_connection_width(steps, edges) == peak
assert query_program_logical_width(program) == 6

bound = bind_register_program(program, ((0, 1), (2, 3)), clean=(4, 5))
circuit = Circuit(6).append(FieldOperation(6, bound.circuit.instructions))
print("Logical peak", peak)
print("Concrete allocated and returned work", [event.kind for event in bound.events])
print("Emitted primitives", dict(Counter(gate.name for gate in circuit.to_ir().instructions)))